# MSN PCT — 从零训练 SkullFix 颅骨补全

取代 `MSN_model_training_Demo.ipynb`。模型拓扑没变（LBR / offset self-attention /
cross-attention 解码器 / copy-and-mapping 上采样），改的是**能不能训得动**。

---

## ⚠️ 运行须知（先看这段）

**1. kernel 必须选 `comp0190-msn`**（右上角 Select Kernel）。这个环境已经装好 TensorFlow 2.15，
以及数据预处理需要的 `scikit-image` / `pynrrd` / `trimesh` / `fpsample`，不用再切换环境。

**2. 单元格必须按顺序跑，尤其是"训练"要在"建模型"之前。**
这个 kernel 一旦建了模型就会占住显存，而训练子进程需要 15.5 GiB / 24 GiB。
顺序反了会 OOM。如果你已经建过模型又想训练，**先 Restart Kernel**。

**3. 数据准备已经做过了**，`data/cache/` 里的缓存可以直接用（下面第 1 节会自检）。
只有当你要改点数或样本量时才需要重跑——用下面新增的「0. 数据准备」那一格，
改一下 `N_SAMPLES` 等参数直接在本 kernel 里跑就行，不用再开终端、也不用切环境。

**4. 训练想跑得更稳的话，用终端而不是 notebook**（断连不会中断）：

```bash
cd /root/comp0190-organ-completion
nohup /root/miniconda3/envs/comp0190-msn/bin/python src/models/train_skullfix.py \
    --minutes 55 > train.log 2>&1 &
tail -f train.log
```

notebook 里的训练单元格做的是同一件事，只是输出直接打在单元格里，关掉页面就没了。

---

## 相对 demo 改了什么

**A. 数据配对错位（正确性 bug）**
`explore_skull.ipynb` 的 `nrrd_to_point_cloud` 对 complete 和 defective **各自独立**做
`normalize_point_cloud`。切掉一块骨头会移动质心、改变最大半径，配对的两朵云于是落到
**不同坐标系**。实测 skull_000：质心偏移 7.55 voxel（半径的 3.6%）、尺度差 2.8%，
同点数下 GT→input 最近距离被抬高 32%。现在从 defective 推出**唯一一个**相似变换同时作用于两者
（推理时只有 defective 可用，所以这个坐标系可复现）。

**B. 距离计算把显存吃爆**
demo 的 `distance_matrix` 把两朵云 tile 成 `(B,N,M,3)`。按 demo 自己的设置（batch 8、6144 点）
光这一个张量前向就 ~12 GB，反向还要留着——这就是推理 notebook 当初被迫退回 CPU 的原因。
换成 `|a|²-2a·b+|b|²` 只产生 `(B,N,M)`。**改完之后完整论文架构（187.5M 参数）在单张 4090 上
372 ms/步、15.5 GiB，训得动了**，不需要缩模型。

**C. DCD 无法从随机初始化引导**
DCD 有界于 [0,2]，两个因子在预测偏离时同时消失。本模型初始化实测：最近邻距离均值 2.79 →
`exp(-2.79)=0.067`；6144 个 GT 点全部坍缩到 **6 个**不同预测点 → 密度权重 ~1/1970。
两者相乘让 loss 钉在 1.9995（上界 2.0），在 1e-7/1e-4/3e-4/1e-3 上扫 40 步 loss 变化不到 0.03。
默认改用 `cd_dcd`：CD 无界负责把形状拉对，DCD 在形状对上后接管精修。

**D. 推理不确定性**
demo 的 `UniformSampler` 用有状态随机数抽质心，**同一个模型对同一输入两次调用输出差 1.03**。
现在训练时保持随机（对 40 个样本相当于免费增广），推理时改用固定种子的 stateless 抽样，
重复调用逐位一致——指标才可复现。

**E. 其它**
- 学习率 1e-7 → 3e-4 + 100 步 warmup。1e-7 配 Adam 比常规小三个数量级。
- batch 8 → 4（24 GB 上 8 会 OOM）。模型里**没有 BatchNorm**（`LBR` 只是 Dense+ReLU），小 batch 只增加梯度噪声。
- `validation_split=0.1` → 按颅骨 id 显式划分。Keras 是**先切尾部再打乱**，一旦每颗颅骨生成多个 partial
  （demo 的 `preprocess_data` 就生成 2 个），兄弟样本会横跨划分边界造成泄漏。
- 冻结的 BERT 预计算。`trainable=False` 且只有 "skull" 一个类别 → 输出是常量，每步重算 1.1 亿参数是浪费。
- checkpoint 存成 `best.h5` 而非 `best.weights.h5`。后者是新版 Keras 格式，即使
  `save_weights_only=True` 也会把 Adam 的动量一起存（187.5M → 562M 个值，**2.25 GB**），
  且每次 val 改善都重写一遍。旧格式只存权重（750 MB）。
- 体素间距。nrrd 头里是各向异性带剪切的 `space directions`（0.451/0.446/0.625 mm），
  demo 在索引空间做 marching cubes，颅骨沿 z 被拉伸约 39%。现在应用该变换并存下 `scale_mm`，
  指标可直接换算成毫米。

## 0. 数据准备（改点数 / 样本量时才需要）

只用 CPU（`skimage` marching cubes + `fpsample` 最远点采样），不碰 GPU，所以和下面的训练 cell 谁先跑都没关系。
跑完后如果 `OUT` 和下面「数据自检」读的缓存路径一致，直接跳到第 1 节验证结果。


In [1]:
import os, subprocess

N_SAMPLES = 0      # 0 = 用 --raw-root 下找到的全部对（当前是 100）；也可以填具体数字
N_DENSE   = 16384
N_IN      = 4096
N_OUT     = 6144
WORKERS   = 8      # measured optimum -- this job is memory-bandwidth bound, not
                   # compute bound, so 12 is slower than 8 and 24 is slower than 4

REPO   = os.path.abspath("../..")
PY_MSN = "/root/miniconda3/envs/comp0190-msn/bin/python"
OUT    = os.path.join(REPO, "data", "cache", f"skullfix_pairs_{N_IN}_{N_OUT}.npz")

proc = subprocess.Popen(
    [PY_MSN, "src/data/prepare_skullfix.py",
     "--n-samples", str(N_SAMPLES),
     "--n-dense", str(N_DENSE),
     "--n-in", str(N_IN),
     "--n-out", str(N_OUT),
     "--workers", str(WORKERS),
     "--out", OUT],
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print("\nreturn code:", proc.returncode)



prepare_skullfix: 100%|██████████| 100/100 [04:28<00:00,  2.69s/it]

saved 100 pairs (0 failed) -> /root/comp0190-organ-completion/data/cache/skullfix_pairs_4096_6144.npz
input (4096, 3)  gt (6144, 3)  scale_mm mean 103.8

return code: 0


## 1. 数据自检

只用 numpy，不碰 GPU。

In [19]:
import os, sys, json, subprocess
import numpy as np

REPO = os.path.abspath("../..")
CACHE = os.path.join(REPO, "data", "cache", "skullfix_pairs_4096_6144.npz")
RUN   = os.path.join(REPO, "experiments", "msn_skullfix")
PY_MSN = "/root/miniconda3/envs/comp0190-msn/bin/python"

assert os.path.exists(CACHE), f"Cache not found. Run prepare_skullfix.py first (see note 3):\n{CACHE}"

data = np.load(CACHE)
ids, inputs, gt = data["ids"], data["inputs"], data["gt"]
scale_mm = float(data["scale_mm"].mean())
print(f"{len(ids)} skull pairs | input {inputs.shape} | gt {gt.shape}")
print(f"Normalisation scale {scale_mm:.1f} mm  (normalised CD x scale_mm = mm)")


def nn_dist(query, ref, chunk=1024):
    """Nearest-neighbour distance from each query point to `ref`. Chunked pure
    numpy -- this env deliberately has no scipy: its versions are pinned tight
    (numpy<2 + tensorflow<2.16) and a diagnostic is not worth touching them."""
    ref2 = (ref ** 2).sum(1)
    out = np.empty(len(query), dtype=np.float64)
    for i in range(0, len(query), chunk):
        q = query[i:i + chunk]
        d2 = (q ** 2).sum(1)[:, None] - 2.0 * (q @ ref.T) + ref2[None, :]
        out[i:i + chunk] = np.sqrt(np.maximum(d2.min(1), 0.0))
    return out


# Pair-alignment self-check: distance from each GT point to the nearest input
# point. The median should sit at roughly the sampling spacing (shared surface)
# and only the tail (the defect) should be large. A large median means the two
# clouds are NOT in the same frame.
nn = nn_dist(gt[0].astype(np.float64), inputs[0].astype(np.float64))
print(f"\nskull_{ids[0]}  GT -> input nearest-neighbour distance:")
print(f"  median {np.median(nn)*scale_mm:6.2f} mm   <- shared surface")
print(f"  p99    {np.percentile(nn,99)*scale_mm:6.2f} mm   <- defect region")
print(f"  fraction beyond 0.05: {(nn>0.05).mean()*100:.1f}%")

100 skull pairs | input (100, 4096, 3) | gt (100, 6144, 3)
Normalisation scale 103.8 mm  (normalised CD x scale_mm = mm)

skull_000  GT -> input nearest-neighbour distance:
  median   2.75 mm   <- shared surface
  p99     20.14 mm   <- defect region
  fraction beyond 0.05: 5.2%


## 2. 训练

**这一格要在建模型之前跑**（本 kernel 还没占显存）。单张 4090 上约 **9 s/epoch**（80 训练 + 20 验证）。

### 什么时候停？——三道闸，实际起作用的是第一道

| 机制 | 默认值 | 说明 |
|---|---|---|
| `EarlyStopping` | patience 20 | **真正的停止信号**，val_loss 连续 20 轮不改善就停，并恢复最佳权重 |
| `ReduceLROnPlateau` | patience 10（由早停推导） | 停滞 10 轮先把 lr 砍半再试。**必须小于早停的 patience，否则永不触发** |
| `--epochs` / `--minutes` | 300 / 90 min | 兜底。**修好学习率衰减后训练明显变长（200~280 轮），300 这个上限已经不宽裕，再延长要先调高** |

### 实测的收敛情况

| run | 配置 | epochs | best val CD_t |
|---|---|---:|---:|
| `baseline_es20` | 默认 | 133 | 7.08 mm |
| `dcd_l2` | `--dcd-lambda 2` | 149 | 6.95 mm |
| `lr_fix_only` | 同上 + **修好的学习率衰减** | 279 | **6.32 mm** |
| `rep_w05` | 同上 + `--repulsion-weight 0.5` | 222 | 6.33 mm |

**最大的单项收益来自修学习率衰减**（6.95 → 6.32 mm），不是任何损失函数改动。
在此之前 `ReduceLROnPlateau` 的 patience 是 40、比早停的 20 大，**从来没有触发过一次**。

`--repulsion-weight` 不改善 CD_t（6.33 vs 6.32，差值远小于 0.49 mm 的运行间方差），
它的作用是**密度**：扎堆率 5.6% → 1.4%。见第 4 节的对比表。

常用参数：`--config small` 换 9.4M 调试版 · `--loss cd` 只用 Chamfer ·
`--repulsion-weight` · `--dcd-lambda` · `--lr` · `--batch-size` · `--n-folds`。

> ⚠️ **`--run-name` 重名会静默覆盖上一轮的 `history.csv` 和 `run.json`。** 重跑同一配置前先改名。
>
> ⚠️ **单折验证只有 20 颗颅骨。** 实测运行间方差约 **0.49 mm**，逐 epoch 抖动 std 约 0.24 mm ——
> **小于 0.5 mm 的 CD_t 差异不要当成改进**。密度指标不受此困扰。

In [20]:
MINUTES = 90    # outer safety net only now -- EarlyStopping(patience=20) does the
                # real stopping (see train_skullfix.py). At ~9.5s/epoch on 100 skulls
                # this is well above the --epochs=300 ceiling (~47 min), so in normal
                # operation neither this nor --epochs should be what actually stops the run.

proc = subprocess.Popen(
    [PY_MSN, "src/models/train_skullfix.py", "--minutes", str(MINUTES),
    "--dcd-lambda", "2",
    "--run-name", "lr_fix_only",],      # ← 加这行，不然会覆盖 baseline 的结果
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print("\nreturn code:", proc.returncode)


2026-08-07 01:47:49.076110: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-07 01:47:49.076127: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-07 01:47:49.076879: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
text feature (768,) (constant: frozen BERT, single class)

config=paper  params=187.5M  in=4096 out=6144
train=80 skulls  val=20 skulls (ids 083, 053, 070, 045, 044...)
dcd_weight=1  dcd_lambda=2
repulsion off
loss=cd_dcd  lr=0.0003  batch=4  budget=90 min  scale=103.8 mm
artifacts -> /root/comp0190-organ-completion/experiments/msn_skullfix/lr_fix_only

Epoch 1/3

## 3. 训练曲线

改用 `src/eval/report.py`，不再在 notebook 里手写画图代码 —— 之前这一节和第 4 节
读的是**两个不同 run**（曲线读 `lr_fix_only`、权重却载入 `dcd_l2`），而曲线标题还在说
"train/validation gap is memorisation"，这个说法早已被实测推翻（val/train 只有 1.03~1.19×）。

`RUNS` 里每加一行，下面所有图表就多一条曲线。

In [ ]:
import pandas as pd
sys.path.insert(0, os.path.join(REPO, "src", "eval"))
sys.path.insert(0, os.path.join(REPO, "src", "models"))
import report as rp

# (显示名, experiments/ 下的相对路径)。带 --run-name 的 run 在 msn_skullfix/ 子目录里。
# 显示名会出现在所有图表里，保持英文。
RUNS = [
    ("baseline", "baseline_es20"),
    ("dcd_l2",   "msn_skullfix/dcd_l2"),
    ("lr_fix",   "msn_skullfix/lr_fix_only"),
    ("rep_w05",  "msn_skullfix/rep_w05"),
]
runs = rp.load_runs(REPO, RUNS)

for r in runs:
    print(f"{r.label:10} {r.config_str():40} "
          f"{r.meta['epochs_run']:4d} epochs (best {r.best_epoch:3d})  "
          f"CD_t {r.meta['best_val_cd_t_mm']:.3f} mm  {len(r.lr_drops)} LR drops")

# 所有 run 共用同一批验证颅骨，否则下面的对比不成立
assert len({tuple(r.meta["val_ids"]) for r in runs}) == 1, \
    "validation splits differ across runs -- not comparable"
print(f"\nValidation set: {len(runs[0].meta['val_ids'])} skulls, identical across all runs")

In [25]:
rp.fig_curves(runs).show()

## 4. 评估 —— 全部 run，全部指标

这里开始才占显存。指标全部由 `report.eval_runs` 计算，每颗验证颅骨一行。

| 指标 | 含义 | 为什么要它 |
|---|---|---|
| **CD_t (mm)** | 双向平均最近邻距离之和 | 主指标。用各颅骨**自己的** `scale_mm` 换算 |
| **HD95 (mm)** | 95 分位的最近邻距离 | **最坏情况**。CD 是均值，会把"某处差 15mm"平均掉，而临床上那正是不可接受的。用 95 分位而非最大值，否则一个离群点就主导了 |
| **F1@0.05 / @0.03** | 阈值内点的精确率/召回率调和平均 | **原论文 Table 1/2 报的就是这两个**，补上才能直接并排。阈值是归一化单位，约合 5.19mm / 3.11mm |
| **DCD** | 密度感知 Chamfer | 统一按 λ=1 报告，与训练用的 λ 无关，保证列可比 |
| **clump_% / spacing_CV** | 最近邻 <2mm 的点占比 / 间距变异系数 | 密度均匀性。GT 分别是 **0.0%** 和 0.145 |

**没有 CD_p。** 它的定义是 `sqrt(平均距离)` —— 对长度开根号，量纲不成立
（这个 bug 从原项目原样继承而来）。不再报告。

In [26]:
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

eval_df = rp.eval_runs(REPO, runs)                       # 每颗颅骨一行
eval_df.to_csv(os.path.join(REPO, "experiments_log", "eval_all_runs.csv"), index=False)
print()
print(rp.format_summary(eval_df))

2026-08-07 03:05:54.541104: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-07 03:05:54.541120: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-07 03:05:54.541766: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


  baseline: 20 skulls
  dcd_l2: 20 skulls
  lr_fix: 20 skulls
  rep_w05: 20 skulls

run                      CD_t_mm     HD95_mm     F1@0.05     F1@0.03         DCD     clump_%  spacing_CV
--------------------------------------------------------------------------------------------------------
baseline                  7.2197      7.2441      0.8619      0.4623      0.8555     12.8459      0.3886
dcd_l2                    7.0309      7.1551      0.8764      0.4762      0.8473     12.4699      0.3857
lr_fix                    6.3996      6.2046      0.9275      0.5559      0.6933      5.6047      0.2974
rep_w05                   6.4086      6.2049      0.9279      0.5558      0.6758      1.3615      0.2383


In [27]:
# 相对 baseline 的变化，所有指标都翻转成「向下 = 变好」
rp.fig_progress(eval_df, baseline="baseline").show()

# 逐颗颅骨的分布。均值会骗人 —— 运行间方差实测 0.49mm，比多数差异都大，
# 箱线图的重叠程度才是「这个差异是否可信」的直接证据。
rp.fig_per_skull(eval_df, "CD_t_mm").show()

## 5. 可视化（点云）

选一个 run 单独载入权重画图。蓝色是预测的完整颅骨，浅红是缺损输入 —— 缺损区应该只有蓝色。

> 散点图只能看"形状对不对"。**要看表面质量和点的疏密，用 `notebooks/MSN_surface_quality.ipynb`**，
> 那边有 mesh 重建、有符号偏差着色和间距诊断图。

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import tensorflow as tf
import msn_skullfix as msn

SHOW_RUN = "rep_w05"          # 改这里换 run；名字来自上面的 RUNS
_run = next(r for r in runs if r.label == SHOW_RUN)

for _g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)
model = msn.build_model(msn.MSNConfig.paper())
model.load_weights(_run.weights)
text_feat = np.load(os.path.join(REPO, "data", "cache", "bert_skull.npy"))

# 用 predict 而不是 model(x) 逐个调用：后者实测每次泄漏 0.29 GiB 且不释放
_val = _run.meta["val_ids"]
val_pos = [int(np.where(ids == v)[0][0]) for v in _val]
preds = model.predict([inputs[val_pos], np.tile(text_feat[None], (len(val_pos), 1))],
                      batch_size=1, verbose=0)
print(f"{SHOW_RUN}: {model.count_params()/1e6:.1f}M parameters, "
      f"{len(preds)} validation skulls inferred")

PRED_COLOR, INPUT_COLOR, GT_COLOR = rp.C_TRAIN, "#EF9A9A", rp.C_GT


def show_completion(k, camera=(1.6, 1.6, 1.2)):
    pred, pos, sid = preds[k], val_pos[k], _val[k]
    fig = go.Figure([
        go.Scatter3d(x=pred[:, 0], y=pred[:, 1], z=pred[:, 2], mode="markers",
                     name="Predicted complete skull",
                     marker=dict(size=1.6, color=PRED_COLOR, opacity=0.85)),
        go.Scatter3d(x=inputs[pos][:, 0], y=inputs[pos][:, 1], z=inputs[pos][:, 2],
                     mode="markers", name="Defective input",
                     marker=dict(size=1.5, color=INPUT_COLOR, opacity=0.45)),
    ])
    fig.update_layout(
        title=f"skull_{sid} ({SHOW_RUN}) — defect region should show blue only",
        height=650,
        legend=dict(itemsizing="constant", x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.6)"),
        scene=dict(aspectmode="data", camera=dict(eye=dict(zip("xyz", camera)))),
        margin=dict(l=0, r=0, b=0, t=40))
    return fig


# 该 run 表现最好的那颗
_best_k = int(eval_df[eval_df["run"] == SHOW_RUN]["CD_t_mm"].reset_index(drop=True).idxmin())
show_completion(_best_k).show()

In [33]:
def show_pred_vs_gt(k):
    pred, pos, sid = preds[k], val_pos[k], _val[k]
    g = gt[pos]
    fig = make_subplots(rows=1, cols=2,
                        specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
                        subplot_titles=(f"Prediction ({SHOW_RUN})", "Ground truth"))
    fig.add_trace(go.Scatter3d(x=pred[:, 0], y=pred[:, 1], z=pred[:, 2], mode="markers",
                               name="Predicted", marker=dict(size=1.4, color=PRED_COLOR)), 1, 1)
    fig.add_trace(go.Scatter3d(x=g[:, 0], y=g[:, 1], z=g[:, 2], mode="markers",
                               name="Ground truth", marker=dict(size=1.4, color=GT_COLOR)), 1, 2)
    fig.update_layout(height=520, title=f"skull_{sid}",
                      legend=dict(itemsizing="constant", orientation="h",
                                  x=0.5, xanchor="center", y=-0.02),
                      scene=dict(aspectmode="data"), scene2=dict(aspectmode="data"))
    return fig


show_pred_vs_gt(_best_k).show()

## 6. 对照组

### 6.1 作者发布的预训练权重

由 `notebooks/MSN_baseline_pretrained.ipynb` 产出，存在
`experiments_log/pretrained_baseline/eval_val20.csv`。**同一批 20 颗验证颅骨、同一套指标定义、
同一份已对齐的 `.npz`**，所以这是一个干净的对照。

⚠️ 性质要写准：这是**"通用基础模型直接应用于颅骨" vs "颅骨专精训练"**，
不是同任务下两个方法的较量。而且论文报的 DCD 是 1.41269，
用它的权重在本项目颅骨上实测是 **1.42772（相差 1%）** —— 说明该模型在颅骨上**并未失效**，
本工作的增益来自**专精化**。这个措辞比"我们修好了它的缺陷"准确得多。

（下面那个旧的 `skullfix_eval_results.csv` 已弃用：它跑在**配对错位的旧 50 样本 .ply** 上。）

### 6.2 原论文报告的数字

| | 训练数据 | CD | DCD | F1@0.05 | F1@0.03 |
|---|---|---:|---:|---:|---:|
| 论文 Table 1（全量） | 200,000 点云 / 240 类 | 0.00170 | 1.41269 | 0.9370 | 0.7408 |
| 论文 Table 2（小数据量） | 4,800 形状 | 0.002327 | 1.60467 | 0.89202 | 0.6234 |

**Table 2 那一行是和本项目处境最接近的参照**（小数据量）。F1 现在可以直接并排比。

⚠️ **CD 那一列不要跨项目比。** 按 `calc_cd` 的线性距离定义，0.00170 会比点间距还小 35 倍，
物理上不可能 —— 推测论文用的是 `loss.py` 里另一个基于平方距离的 `chamfer_distance_loss`。
仓库里没有评测脚本，无法确证。**DCD 和 F1 没有这个问题。**

In [ ]:
pre = pd.read_csv(os.path.join(REPO, "experiments_log", "pretrained_baseline", "eval_val20.csv"))
best_run = eval_df.groupby("run")["CD_t_mm"].mean().idxmin()
mine = eval_df[eval_df["run"] == best_run]

print(f"Same {len(pre)} validation skulls, same metric definitions\n")
print(f"{'':<46}{'CD_t (mm)':>12}{'DCD':>10}")
print("-" * 68)
print(f"{'Released pretrained weights (not skull-trained)':<46}"
      f"{pre['CD_t_mm'].mean():>12.3f}{pre['DCD'].mean():>10.4f}")
print(f"{'This work (' + best_run + ', trained from scratch)':<46}"
      f"{mine['CD_t_mm'].mean():>12.3f}{mine['DCD'].mean():>10.4f}")
print("-" * 68)
print(f"{'Improvement':<46}"
      f"{(1 - mine['CD_t_mm'].mean() / pre['CD_t_mm'].mean()) * 100:>11.1f}%"
      f"{(1 - mine['DCD'].mean() / pre['DCD'].mean()) * 100:>9.1f}%")
print(f"\nSanity check: the source paper reports DCD = 1.41269 (Table 1); measured here "
      f"{pre['DCD'].mean():.5f}\n  -> {abs(pre['DCD'].mean() - 1.41269) / 1.41269 * 100:.1f}% "
      f"difference. Metric implementation and weight loading both verified.")

## 下一步

按目前的证据强度排序（信噪比高的优先）：

1. **去掉 DCD 的消融**：`--loss cd --repulsion-weight 0.5`。
   DCD 的密度项来自 `argmin`，**梯度实测为 None**，结构上无法推开点；密度现已由 repulsion 接管；
   而 `dcd_lambda` 的调参收益又小于噪声。**若去掉它结果不变，就能把三个损失砍成两个** ——
   这比继续叠加损失项是更强的结论。
2. **量一下缺损区占 GT 的比例**。目前 CD_t 主要在衡量"有没有把可见区域抄好"，
   缺损区被稀释了。这个数决定要不要走"只预测 implant"那条路
   （`data/.../training_set/implant/` 的 100 个文件一直没用过）。十几行代码，不用训练。
3. **缺损区限定的 CD / HD95 / F1**。做完 2 就能顺手做，**这才是论文该报的主指标**。
4. **k 折交叉验证**。现在所有结论都是单折 20 颗，运行间方差 0.49mm。
   代码（`--n-folds`）早就就位，只是没跑。**定稿前必须做，且要留足时间（5 折 ≈ 5 倍）。**

暂不做：体素基线（范围已与导师确认延后）、平滑惩罚项（缺可信的粗糙度度量，
详见 devlog 2026-08-07 补充记录）、正则化（已试并否决，模型不过拟合）。